# Demo B — 15-model YOLO consensus with model-vote filtering (final submission, public LB 0.39)

Reproduces Kaggle submission ref 57782016 (public PQ 0.39; 1,259 instances over 177 of the 180 test images).

Inputs this notebook needs (both public, no challenge)
1. the competition data (attach this competition): test/test_images/*.jpeg;
2. the public weights dataset notoverkil/solar-filament-weights (YOLO11m / YOLO26m checkpoints in p1-*, p2-*,
   p3-* families). The same dataset also contains an unrelated submitted CSV, which we do NOT use.

Pipeline shown: preprocessing (2048 canvas; 8 dihedral TTA views) -> per-model instance inference with the
published checkpoints (no training required) -> consensus over all model/view instances (greedy IoU >= 0.5
clustering, score-weighted pixel vote >= 0.5, min area 150 px, max 150 detections, greedy de-overlap, COCO RLE)
-> our model-vote filter (keep instances supported by all 15 models with IoU > 0.5) -> validation.

Runtime about 60-90 min for the full ensemble on one T4; the script supports sharding. Our code is MIT.

In [ ]:
import glob, os
from pathlib import Path

REPO = Path.cwd()
INPUT = Path('/kaggle/input')
tests = [p for p in glob.glob(str(INPUT / '**' / 'test_images'), recursive=True) if os.path.isdir(p)]
ckpts = glob.glob(str(INPUT / '**' / '*.pt'), recursive=True)
print('test_images:', tests)
print('checkpoints found:', len(ckpts))
assert tests, 'attach this competition (test/test_images)'
assert ckpts, 'attach the public weights dataset notoverkil/solar-filament-weights'

In [ ]:
# 1) Per-model inference + consensus, shard 0 of 2 (keeps each session inside the GPU time limit).
!python experiments/code/cloud_notoverkil_ensemble.py --mode test --shard-index 0 --shard-count 2 --families yolo11m-1536,yolo11m-2048,yolo26m-1536

In [ ]:
# 2) shard 1 of 2 (optionally in a second session; its cache dir is passed to assemble below).
!python experiments/code/cloud_notoverkil_ensemble.py --mode test --shard-index 1 --shard-count 2 --families yolo11m-1536,yolo11m-2048,yolo26m-1536

In [ ]:
# 3) Consensus over the cached instances -> unfiltered submission (2,328 instances, 180 images).
import glob
from pathlib import Path
caches = sorted({str(Path(p).parent) for p in glob.glob('/kaggle/working/solar_ensemble/cache_test_shard*/*.npz')})
print('cache dirs:', caches)
joined = ' '.join(caches)
!python experiments/code/cloud_notoverkil_ensemble.py --mode assemble --cache-dirs $joined --submit

In [ ]:
# 4) Our model-vote filter: keep only instances supported by all 15 models (IoU > 0.5 per model).
# assets/nk_v3_unfiltered_consensus.csv and assets/nk_v3_consensus_support.npz ship the exact row-aligned
# intermediate files of the submitted run, so this step is reproducible without re-running inference.
!python scripts/vote_filter_nk.py --unfiltered-csv assets/nk_v3_unfiltered_consensus.csv --support-npz assets/nk_v3_consensus_support.npz --threshold 15 --out submission_vote15.csv

In [ ]:
# 5) Validate and summarise (expected: 1,259 instances over 177 images, pixel-disjoint).
!python src/validate_submission.py submission_vote15.csv
import pandas as pd
df = pd.read_csv('submission_vote15.csv')
stems = df['filament_id'].str.rsplit('_', n=1).str[0]
print('rows:', len(df), '| images:', stems.nunique())

Optional out-of-fold evaluation of the same consensus pipeline on the harness fold-0 split (one model per
family, honest OOF): run

    !python experiments/code/cloud_notoverkil_ensemble.py --mode fold0-eval --families all

Expected numbers for the submitted file: 1,259 instances / 177 images, public LB 0.39 (ref 57782016). The
unfiltered consensus has 2,328 instances / 180 images and scores 0.38-0.39; the vote filter mostly removes
low-support instances rather than changing the score.